# Tech Challenge IADT Fase 4 - Sistema Multimodal Completo

# Apresentacao: 15 minutos

## Objetivo

Este projeto implementa um **sistema de análise multimodal** para monitoramento médico, conforme o Tech Challenge da Fase 4.
Ele integra três fontes de dados e as combina em uma única decisão clínica:

- **Vídeo** (sessões de fisioterapia/cirurgia): postura com **OpenPose‑Light (MediaPipe)** + objetos/áreas críticas com **YOLOv8**.
- **Áudio** (consultas médicas): transcrição e análise de sentimento/entidades com **Azure Cognitive Services** (Speech to Text + Text Analytics), com detecção de alterações vocais (fadiga, falta de ar, dor).
- **Anomalias**: séries temporais de sinais vitais, evolução de prescrições e padrões de movimentação.

Os três módulos alimentam um **workflow integrado** que calcula uma **pontuação de risco** e gera o **alerta final à equipe médica**. O notebook roda em dois modos (escolhidos na execução): **offline** (dados simulados, sem chaves) ou **azure** (serviços reais).

In [ ]:
# =====================================================================
# SEÇÃO 0 — PREPARAÇÃO DO AMBIENTE E MODO DE EXECUÇÃO
# Nesta abertura nós carregamos as bibliotecas de visão computacional e
# dados que serão usadas ao longo da apresentação e definimos em qual
# modo o sistema vai rodar (offline para demonstração ou Azure real).
# =====================================================================
import cv2          # leitura e manipulação de vídeo/frames (Aula 01)
import numpy as np  # operações numéricas sobre imagens e séries temporais
import torch        # carrega o modelo YOLOv8 (Aulas de visão computacional)
import mediapipe as mp  # OpenPose-Light: análise postural (Aulas 02/03)
import pandas as pd # tabelas de sinais vitais (séries temporais)
from scipy import stats # apoio estatístico para detecção de anomalias
import os           # variáveis de ambiente e caminhos de arquivo

# ---- Configuracao de execucao (switch em tempo de execucao) ----
# Este é o ponto onde, na demonstração, informamos a origem dos dados:
# MODO = "offline"  -> sem Azure: usa transcricao/texto e frames simulados
# MODO = "azure"    -> usa Azure Cognitive Services (exige chaves e pacotes)
try:
    MODO = input("Modo de execucao [offline/azure] (Enter = offline): ").strip().lower() or "offline"
except Exception:
    MODO = os.getenv("MODO", "offline")
CAMINHO_VIDEO = os.getenv("CAMINHO_VIDEO", "dados/video.mp4")  # vídeo clínico real (opcional)
CAMINHO_AUDIO = os.getenv("CAMINHO_AUDIO", "dados/audio.wav")  # áudio de consulta (opcional)
AZURE_ON = (MODO == "azure")
print("MODO de execucao:", MODO)

# ---- MediaPipe Pose (OpenPose-Light) ----
# Inicializamos o detector de postura uma única vez; se a biblioteca não
# estiver disponível, seguimos em modo offline sem travar a apresentação.
try:
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(
        static_image_mode=False,     # modo de vídeo (rastreio entre frames)
        model_complexity=1,          # equilíbrio entre precisão e velocidade
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    _mp_ok = True
except Exception:
    pose = None
    _mp_ok = False

In [ ]:
# =====================================================================
# SEÇÃO 0b — LOG DE EXECUÇÃO
# Tudo que as APIs/etapas retornam fica registrado em 3 lugares: no console,
# num arquivo logs/execucao_<timestamp>.log e na lista REGISTRO_API (exibida
# por inteiro no final) para você auditar cada retorno.
# =====================================================================
import logging, datetime, json as _json

LOG_DIR = "logs"
os.makedirs(LOG_DIR, exist_ok=True)
LOG_FILE = os.path.join(LOG_DIR, "execucao_%s.log" % datetime.datetime.now().strftime("%Y%m%d_%H%M%S"))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    handlers=[logging.FileHandler(LOG_FILE, encoding="utf-8"), logging.StreamHandler()],
)
log = logging.getLogger("techchallenge")
log.info("===== INICIO DA EXECUCAO | log em: %s =====", LOG_FILE)

REGISTRO_API = []  # [(nome, dado)] de cada chamada/passo

def _resumir(dado, limite=400):
    try:
        t = _json.dumps(dado, ensure_ascii=False, default=str)
    except Exception:
        t = str(dado)
    return t if len(t) <= limite else t[:limite] + "...(+%d chars)" % (len(t) - limite)

def registrar_api(nome, dado):
    """Registra o retorno bruto de uma API/passo: ecoa no log e acumula p/ o resumo final."""
    REGISTRO_API.append({"nome": nome, "dado": dado})
    log.info("[RETORNO] %s -> %s", nome, _resumir(dado))
    return dado

In [ ]:
# ---------------------------------------------------------------------
# Modelo YOLOv8 — detecção de objetos e áreas críticas no vídeo clínico.
# Maneira OFICIAL de carregar o YOLOv8 (a lib 'ultralytics' já está
# instalada):  from ultralytics import YOLO ; YOLO('yolov8n.pt')
# OBS: 'torch.hub.load("ultralytics/yolov8", ...)' NÃO funciona — esse repo
# não existe no GitHub (404) e o YOLOv8 não é distribuído via torch.hub.
# Na 1ª execução os pesos 'yolov8n.pt' são baixados e depois ficam em cache.
# Se o download falhar (ex.: sem internet), seguimos em modo simulado.
# ---------------------------------------------------------------------
try:
    from ultralytics import YOLO
    model_yolo = YOLO('yolov8n.pt')
except Exception as e:
    print("YOLO indisponivel -> usarei objetos simulados:", e)
    model_yolo = None
_yolo_online = model_yolo is not None  # flag que controla real x simulado
print("YOLO disponivel:", _yolo_online)

In [ ]:
def detectar_objetos_yolo(frame):
    """
    Passa um frame pelo YOLOv8 e devolve a lista de objetos detectados
    (rótulo, confiança e caixa delimitadora). Na apresentação, serve para
    mostrar a detecção de instrumentos críticos e áreas de risco.
    API do ultralytics: model(frame) -> list[Results]; boxes em Results[0].
    """
    if _yolo_online and model_yolo is not None:
        resultados = model_yolo(frame)            # lista de Results (1 por imagem)
        objetos = []
        caixas = resultados[0].boxes              # detecções do primeiro frame
        for caixa in caixas:
            x1, y1, x2, y2 = caixa.xyxy[0].tolist()   # caixa delimitadora
            conf = float(caixa.conf[0])              # confiança da detecção
            cls = int(caixa.cls[0])                  # índice da classe (COCO)
            label = model_yolo.names[cls]            # nome legível da classe
            objetos.append({'label': label, 'confidence': conf,
                            'bbox': [int(x1), int(y1), int(x2), int(y2)]})
        return objetos
    else:
        # Modo offline: objetos simulados para demostracao
        # (garante que a dinâmica do pipeline funcione sem o modelo real)
        obj1 = {'label': 'instrumento_critico', 'confidence': 0.85, 'bbox': [100, 100, 200, 300]}
        obj2 = {'label': 'area_perigosa', 'confidence': 0.72, 'bbox': [300, 50, 400, 200]}
        return [obj1, obj2]

In [ ]:
def detectar_postura(frame):
    """
    Analisa a postura do paciente/profissional usando MediaPipe Pose
    (alternativa leve ao OpenPose). Retorna landmarks 3D de pontos-chave
    (joelhos, quadril, ombros etc.) usados depois para identificar
    movimentos fora do padrão clínico. Retorna SEMPRE um dict.
    """
    if _mp_ok and pose is not None:
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # MediaPipe usa RGB
        results = pose.process(image_rgb)
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            keypoints = {}
            nomes = ['nariz', 'ldeltoide', 'rdeltoide', 'lquadril', 'rquadril',
                     'ljoelho', 'rjoelho', 'ltobillo', 'rtobillo']
            for name, lm in zip(nomes, landmarks):        # guarda x,y,z de cada ponto
                keypoints[name] = (lm.x, lm.y, lm.z)
            return {'status': 'sucesso', 'keypoints': keypoints}
    return {'status': 'nenhum_corpo_detectado'}

In [ ]:
def processar_video_clinico(caminho_video, max_frames=50):
    """
    Percorre o vídeo frame a frame combinando as duas análises de vídeo:
    YOLOv8 (objetos críticos) + MediaPipe (postura). Para cada frame gera
    um registro com score de anomalia, alertas e os keypoints de postura
    (campo 'kpts'), que depois alimentam a análise de movimentação.
    """
    cap = cv2.VideoCapture(caminho_video)  # abre o arquivo de vídeo
    resultados = []
    frame_count = 0
    while cap.isOpened() and frame_count < max_frames:  # limita frames na demo
        ret, frame = cap.read()
        if not ret:
            break
        objetos = detectar_objetos_yolo(frame)  # objetos/áreas críticas
        postura = detectar_postura(frame)       # landmarks posturais
        score_anomalia = 0
        alertas = []
        kpts = None
        if postura['status'] == 'sucesso':
            kpts = postura['keypoints']
            lje = kpts.get('ljoelho', (0,0,0))
            rje = kpts.get('rjoelho', (0,0,0))
            # Regras simples de postura inadequada (joelho muito esticado/dobrado)
            if lje[2] > 0.8 or rje[2] > 0.8:
                score_anomalia += 1
                alertas.append('Joelho esticada')
            if lje[2] < 0.2 or rje[2] < 0.2:
                score_anomalia += 1
                alertas.append('Joelho dobrado excessivamente')
        if len(objetos) > 0:
            # Considera críticos apenas os objetos com confiança > 0.5
            labels_criticos = [o['label'] for o in objetos if o['confidence'] > 0.5]
            alertas.append('Objetos detectados: %s' % ', '.join(labels_criticos[:3]))
            score_anomalia += len(labels_criticos)
        resultados.append({'frame': frame_count, 'objetos': len(objetos), 'postura_ok': postura['status'] == 'sucesso', 'score_anomalia': score_anomalia, 'alertas': '; '.join(alertas) if alertas else 'Nenhum', 'kpts': kpts})
        frame_count += 1
    cap.release()
    return resultados

## 2. Analise de Audio Medico (5 min)

In [ ]:
# =====================================================================
# SEÇÃO 2 — ANÁLISE DE ÁUDIO (AZURE COGNITIVE SERVICES)
# Configuramos Speech to Text (transcrição) e Text Analytics
# (sentimento/entidades). Tudo protegido: sem libs/chaves, o sistema cai
# em modo offline e a demonstração continua.
# =====================================================================
# Carrega um arquivo .env, se existir (python-dotenv). Garante que as chaves
# do Azure cheguem ao kernel do Jupyter sem depender de 'export' no shell.
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

try:
    import azure.cognitiveservices.speech as speechsdk      # transcrição de fala
    from azure.ai.textanalytics import TextAnalyticsClient  # sentimento + entidades
    from azure.core.credentials import AzureKeyCredential
    _HAS_AZURE_LIBS = True
except Exception:
    _HAS_AZURE_LIBS = False

# Credenciais vêm do ambiente (ou do .env) — nunca hardcoded.
speech_key = os.getenv("AZURE_SPEECH_KEY", "")
speech_region = os.getenv("AZURE_SPEECH_REGION", "westus2")
text_key = os.getenv("AZURE_TEXT_ANALYTICS_KEY", "")
text_endpoint = os.getenv("AZURE_TEXT_ANALYTICS_ENDPOINT", "").strip()

# _azure_on = true apenas se MODO=azure E libs E chaves presentes
_azure_on = bool(AZURE_ON and _HAS_AZURE_LIBS and speech_key and text_key)
if _azure_on:
    speech_config = speechsdk.SpeechConfig(subscription=speech_key, region=speech_region)
    speech_config.speech_recognition_language = "pt-BR"
    text_analytics_client = TextAnalyticsClient(
        endpoint=text_endpoint,
        credential=AzureKeyCredential(text_key),
    )
else:
    speech_config = None
    text_analytics_client = None

# ---- Diagnostico EXPLICITO do que habilita (ou nao) o Azure ----
print("--- Condicao do Azure ---")
print("  pacotes Azure instalados :", _HAS_AZURE_LIBS)
print("  MODO == 'azure'          :", AZURE_ON)
print("  AZURE_SPEECH_KEY definida:", bool(speech_key))
print("  AZURE_TEXT_ANALYTICS_KEY :", bool(text_key))
print("  Endpoint Text Analytics  :", text_endpoint,
      "(do .env)" if text_endpoint else "(legado por regiao)")
if _azure_on:
    print("=> Azure LIGADO (Speech to Text + Text Analytics)")
else:
    faltas = []
    if not AZURE_ON:
        faltas.append("MODO=offline (nao pediu azure)")
    if not _HAS_AZURE_LIBS:
        faltas.append("pacotes Azure ausentes (pip install azure-cognitiveservices-speech azure-ai-textanalytics)")
    if not speech_key:
        faltas.append("falta AZURE_SPEECH_KEY")
    if not text_key:
        faltas.append("falta AZURE_TEXT_ANALYTICS_KEY")
    print("=> Azure DESLIGADO, usando OFFLINE. Motivos:", "; ".join(faltas) or "-")

In [ ]:
def transcrever_audio(caminho_audio):
    """
    Converte o áudio da consulta médica em texto (fala em texto, Aula 04)
    usando o Azure Speech to Text. Offline/sem arquivo -> string vazia, e
    o passo de execução usa uma transcrição simulada de exemplo.
    Cada retorno é registrado no log.
    """
    if not _azure_on:
        registrar_api("Azure Speech", {"status": "skipped", "motivo": "_azure_on=False", "texto": ""})
        return ""
    try:
        audio_config = speechsdk.AudioConfig(filename=caminho_audio)
        speech_recognizer = speechsdk.SpeechRecognizer(
            speech_config=speech_config, audio_config=audio_config)
        result = speech_recognizer.recognize_once()
        motivo = str(result.reason)
        if result.reason == speechsdk.ResultReason.RecognizedSpeech:
            registrar_api("Azure Speech", {"status": "ok", "arquivo": caminho_audio,
                                           "reason": motivo, "texto": result.text})
            print(f"Transcricao do audio ({caminho_audio}): {result.text}")
            return result.text  # texto transcrito da fala
        registrar_api("Azure Speech", {"status": "sem_fala", "arquivo": caminho_audio, "reason": motivo})
        return ""
    except Exception as e:
        registrar_api("Azure Speech", {"status": "erro", "arquivo": caminho_audio, "erro": str(e)})
        print(f"Erro na transcricao: {e}")
        return ""

In [ ]:
def analisar_texto_azure(texto):
    """
    Analisa o texto transcrito: sentimento (positivo/neutro/negativo) e
    extração de entidades relevantes. No Azure usa Text Analytics; offline
    aplica uma heurística simples por palavras-chave clínicas.
    Cada retorno é registrado no log (registrar_api) para auditoria.
    """
    padrao = {'sentimento': {'rotulo': 'neutro',
                             'pontuacoes': {'positivo': 0.0, 'negativo': 0.0, 'neutro': 1.0}},
              'entidades': []}
    if not texto or len(texto.strip()) < 3:
        registrar_api("Text Analytics", {"status": "skipped", "motivo": "texto vazio/curto"})
        return padrao
    if not _azure_on:
        t = texto.lower()
        negativos = ['dor', 'cansado', 'fadiga', 'falta de ar', 'exausto',
                     'complicacao', 'desconforto', 'socorro']
        neg = any(w in t for w in negativos)
        resultado = {'sentimento': {'rotulo': ('negativo' if neg else 'neutro'),
                                    'pontuacoes': {'positivo': 0.0,
                                                   'negativo': (1.0 if neg else 0.0),
                                                   'neutro': (0.0 if neg else 1.0)}},
                     'entidades': []}
        registrar_api("Text Analytics (offline)", {"status": "heuristica", **resultado})
        print(f"Analise de texto (offline): {resultado['sentimento']['rotulo']}, entidades: {resultado['entidades']}")
        return resultado
    try:
        sentiment_result = text_analytics_client.analyze_sentiment([texto])[0]
        entity_result = text_analytics_client.recognize_entities([texto])[0]
    except Exception as e:
        msg = str(e)
        registrar_api("Text Analytics (Azure)", {"status": "erro", "erro": msg})
        print(f"Azure Text Analytics indisponivel: {msg}")
        if "unauthorized" in msg.lower() or "401" in msg:
            print("  -> 401: chave invalida OU endpoint legado num recurso novo. "
                  "Preencha AZURE_TEXT_ANALYTICS_ENDPOINT com o endpoint do recurso.")
        if "forbidden" in msg.lower() or "403" in msg:
            print("  -> 403 Forbidden: endpoint de outro produto (ex.: ...search.windows.net "
                  "= Azure AI SEARCH) ou firewall do recurso bloqueando. Use o endpoint do "
                  "recurso 'Azure AI Language' e/ou libere a rede em 'Networking'.")
        return padrao
    # Filtra apenas categorias de interesse clínico
    entidades_medicas = []
    for entity in entity_result.entities:
        if entity.category in ['Person', 'Organization', 'Quantity', 'Temperature', 'DateTime']:
            entidades_medicas.append({'texto': entity.text, 'tipo': entity.category,
                                      'confianca': entity.confidence_score})
    cs = sentiment_result.confidence_scores
    resultado = {'sentimento': {'rotulo': sentiment_result.sentiment,
                                'pontuacoes': {'positivo': float(cs.positive),
                                               'negativo': float(cs.negative),
                                               'neutro': float(cs.neutral)}},
                 'entidades': entidades_medicas}
    registrar_api("Text Analytics (Azure)", {"status": "ok", **resultado})
    return resultado

In [ ]:
def detectar_alteracoes_vocais(texto_transcrito):
    """
    Procura no texto da fala sinais indicativos de condições médicas
    (fadiga, dificuldade respiratória, dor), conforme o Tech Challenge.
    Retorna, por condição, se foi encontrada e com quais termos.
    """
    padroes_clinicos = {
        'fadiga': ['cansado', 'fadiga', 'sono', 'exausto'],
        'dificuldade_respiratoria': ['dificuldade para respirar', 'falta de ar'],
        'dor': ['dor', 'complicacao'],
    }
    texto_lower = texto_transcrito.lower() if texto_transcrito else ''
    resultados = {}
    for condicao, palavras in padroes_clinicos.items():
        encontradas = [p for p in palavras if p in texto_lower]  # casa termos-chave
        resultados[condicao] = {'presente': len(encontradas) > 0, 'encontradas': encontradas, 'quantidade': len(encontradas)}
    return resultados

In [ ]:
def detectar_anomalias_sinais_vitais(df_sinais):
    """
    Detecção de anomalias em séries temporais de sinais vitais.
    Usa o método "3 sigma" (média ± 3 desvios) para frequência cardíaca
    e um limite clínico fixo (SpO2 < 95%) para oxigenação — exatamente os
    exemplos citados no enunciado (batimentos e oxigenação).
    """
    anomalia = {'fc_anomalia': False, 'bp_anomalia': False, 'spo2_anomalia': False, 'alertas': []}
    if 'fc' in df_sinais.columns:
        fc = df_sinais['fc']
        fc_media = fc.mean()
        fc_desvio = fc.std()
        # Faixa de normalidade estatística (3 sigma)
        fc_limite_superior = fc_media + (3 * fc_desvio)
        fc_limite_inferior = fc_media - (3 * fc_desvio)
        fc_atual = df_sinais['fc'].iloc[-1]  # leitura mais recente
        if fc_atual > fc_limite_superior:
            anomalia['fc_anomalia'] = True
            anomalia['alertas'].append('FC CRITICA: %.1f bpm' % fc_atual)
        elif fc_atual < fc_limite_inferior:
            anomalia['fc_anomalia'] = True
            anomalia['alertas'].append('FC CRITICA: %.1f bpm' % fc_atual)
        else:
            anomalia['alertas'].append('FC normal: %.1f bpm' % fc_atual)
    if 'spo2' in df_sinais.columns:
        spo2 = df_sinais['spo2']
        spo2_atual = spo2.iloc[-1]
        # Hipoxemia: oxigenação abaixo de 95% é sinal de risco
        if spo2_atual < 95:
            anomalia['spo2_anomalia'] = True
            anomalia['alertas'].append('SpO2 BAIXA: %.1f%%' % spo2_atual)
        else:
            anomalia['alertas'].append('SpO2 normal: %.1f%%' % spo2_atual)
    return anomalia

In [ ]:
def detectar_anomalias_prescricoes(lista_prescricoes):
    """
    Compara a prescrição atual com a anterior para identificar alterações
    inesperadas no tratamento: novo medicamento, mudança de dosagem e um
    possível risco de interação quando ambos ocorrem juntos.
    """
    anomalia = {'medicamento_novo': False, 'dosagem_alterada': False, 'interacao_drogas': False, 'alertas': []}
    if len(lista_prescricoes) < 2:
        anomalia['alertas'].append('Historico insuficiente')
        return anomalia
    atual = lista_prescricoes[-1]
    anterior = lista_prescricoes[-2]
    # Compara por nome normalizado, mas exibe/usa os nomes originais.
    mapa_atual = {k.lower(): (k, v) for k, v in atual.items()}
    mapa_anterior = {k.lower(): (k, v) for k, v in anterior.items()}
    novos = set(mapa_atual) - set(mapa_anterior)
    if novos:
        anomalia['medicamento_novo'] = True
        nomes = ', '.join(mapa_atual[n][0] for n in sorted(novos))
        anomalia['alertas'].append('Novo medicamento: %s' % nomes)
    comuns = set(mapa_atual) & set(mapa_anterior)
    for med in sorted(comuns):
        # dosagem mudou para o mesmo medicamento
        if mapa_atual[med][1] != mapa_anterior[med][1]:
            anomalia['dosagem_alterada'] = True
            anomalia['alertas'].append('Dosagem alterada para: %s (%s -> %s)'
                                        % (mapa_atual[med][0], mapa_anterior[med][1], mapa_atual[med][1]))
    # Novo medicamento + mudança de dosagem levantam suspeita de interação
    if anomalia['medicamento_novo'] and anomalia['dosagem_alterada']:
        anomalia['interacao_drogas'] = True
        anomalia['alertas'].append('WARNING: Possivel interacao: novo medicamento + cambio de dosagem')
    return anomalia

In [ ]:
def detectar_anomalias_movimentacao(keypoints_history):
    """
    Analisa padrões de movimentação do paciente a partir dos keypoints de
    postura ao longo do tempo. Mede o deslocamento do centroide do quadril
    frame a frame e sinaliza movimento repentino, imobilidade anormal e o
    padrão suspeito de queda (movimento brusco seguido de imobilidade).
    """
    anomalia = {'movimento_repetino': False, 'imobilidade_anormal': False, 'quedas_suspeitas': False, 'alertas': []}
    if len(keypoints_history) < 2:
        anomalia['alertas'].append('Historico insuficiente')
        return anomalia
    # Centroide (ponto médio) entre os dois quadris = proxy da posição do corpo
    centoidos = []
    for kp_data in keypoints_history:
        lq = kp_data.get('lquadril', (0, 0))
        rq = kp_data.get('rquadril', (0, 0))
        centroide = ((lq[0] + rq[0]) / 2, (lq[1] + rq[1]) / 2)
        centoidos.append(centroide)
    if len(centoidos) >= 2:
        # Distância percorrida entre frames consecutivos = velocidade relativa
        velocidades = []
        for i in range(1, len(centoidos)):
            dist = ((centoidos[i][0] - centoidos[i-1][0])**2 + (centoidos[i][1] - centoidos[i-1][1])**2)**0.5
            velocidades.append(dist)
        if velocidades:
            velocidade_max = max(velocidades)
            velocidade_media = sum(velocidades) / len(velocidades)
            # Pico muito acima da média indica movimento repentino
            if velocidade_max > 3 * velocidade_media and velocidade_media > 0:
                anomalia['movimento_repetino'] = True
                anomalia['alertas'].append('Movimento repentino: dist %.3f' % velocidade_max)
            # Pouco movimento nas últimas leituras => imobilidade anormal
            N = min(10, len(velocidades))
            ultimas = velocidades[-N:]
            vm = sum(ultimas) / len(ultimas) if ultimas else 0
            if vm < 0.05 and len(ultimas) >= 10:
                anomalia['imobilidade_anormal'] = True
                anomalia['alertas'].append('Imobilidade: %d frames sem movimento' % len([v for v in ultimas if v < 0.01]))
            # Combinação dos dois sugere possível queda
            if anomalia['movimento_repetino'] and anomalia['imobilidade_anormal']:
                anomalia['quedas_suspeitas'] = True
                anomalia['alertas'].append('WARNING: Padrao de queda suspeito: movimento repentino + imobilidade')
    return anomalia

In [ ]:
def workflow_integrado(resultado_video, resultado_audio, resultado_sinais):
    """
    INTEGRAÇÃO MULTIMODAL — consolida vídeo + áudio + anomalias em uma
    única decisão clínica. Cada fonte contribui com pontos de risco
    (vídeo=2 por alerta, áudio=3 por alteração, sinais=3), e a soma é
    convertida em um nível (Normal / ALTO / CRÍTICO) com ação recomendada.
    """
    alertas_gerais = []
    punteis_risco = 0
    # --- contribuição do VÍDEO: alertas de objetos/postura ---
    if resultado_video and 'total_alertas' in resultado_video:
        vs = resultado_video.get('alertas_listas', [])
        sig = [a for a in vs if 'Nenhum' not in a]
        if sig:
            alertas_gerais.extend(sig)
            punteis_risco += len(sig) * 2
    # --- contribuição do ÁUDIO: alterações vocais detectadas ---
    if resultado_audio and 'total_alteracoes' in resultado_audio:
        aps = [k for k, v in resultado_audio['alteracoes'].items() if v.get('presente', False)]
        if aps:
            # Traduz cada código de alteração em uma mensagem legível para a equipe
            mapa = {'fadiga': 'Paciente relata fadiga', 'dificuldade_respiratoria': 'Paciente relata dificuldade respiratoria', 'dor': 'Paciente relata dor', 'alerta_urgencia': 'ALERTA URGENCIA: solicitacao de ajuda'}
            for a in aps:
                m = mapa.get(a, a)
                alertas_gerais.append(m)
                punteis_risco += 3
    # --- contribuição dos SINAIS VITAIS/prescrições/movimentação ---
    if resultado_sinais and 'alertas' in resultado_sinais:
        alertas_gerais.extend(resultado_sinais['alertas'])
        punteis_risco += 3
    # --- decisão final pela pontuação de risco acumulada ---
    if punteis_risco >= 6:
        decisao = 'ALERTA MÉDICO IMEDIATO'
        nivel = 'CRITICO'
        acao = 'Contatar equipe médica imediatamente'
    elif punteis_risco >= 3:
        decisao = 'ALERTA - Revisao necessaria'
        nivel = 'ALTO'
        acao = 'Avaliar paciente e monitorar 30 min'
    else:
        decisao = 'Monitoramento normal continuo'
        nivel = 'Normal'
        acao = 'Continuar rotina de cuidados'
    return {'decisao': decisao, 'nivel': nivel, 'acao_recomendada': acao, 'total_alertas': len(alertas_gerais), 'alertas': '; '.join(alertas_gerais) if alertas_gerais else 'Nenhum motivo de alerta', 'punteis_risco': punteis_risco}

## 3. Captura / seleção da fonte dos arquivos
Na execução, cada mídia (vídeo e áudio) pode: **(1)** ser **capturada** ao vivo do
dispositivo (parar ao teclar), **salvando** em `capturas/` e seguindo para análise;
**(2)** receber um **caminho** de arquivo existente; ou **(3)** usar o **mock simulado**
(offline). A escolha define `CAMINHO_VIDEO`/`CAMINHO_AUDIO` que o pipeline consome a seguir.
No modo offline o áudio capturado é transcrito de verdade com **Vosk (pt-BR)**; no modo
Azure usa-se o **Speech to Text**.

In [ ]:
# =====================================================================
# SEÇÃO 3 — CAPTURA / SELEÇÃO DA FONTE DOS ARQUIVOS (vídeo e áudio)
# - capturar: grava do dispositivo e salva em capturas/ (para ao teclar)
# - caminho : usa um arquivo que você informar
# - simulado: não cria arquivo -> o pipeline usa dados mockados
# Tudo em try/except: sem câmera/microfone a opção cai em "simulado",
# garantindo que o notebook também rode em modo headless.
# =====================================================================
import wave, datetime

PASTA_CAPTURAS = "capturas"
os.makedirs(PASTA_CAPTURAS, exist_ok=True)

# ---------- Captura de VÍDEO (webcam) ----------
def capturar_video(fonte=0, fps=20):
    """Grava da webcam até a tecla 'q' na janela; salva um .mp4 e retorna o caminho."""
    cap = cv2.VideoCapture(fonte)
    if not cap.isOpened():
        raise RuntimeError("camera indisponivel")
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) or 640
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 480
    saida = os.path.join(PASTA_CAPTURAS,
                         "video_%s.mp4" % datetime.datetime.now().strftime("%Y%m%d_%H%M%S"))
    writer = cv2.VideoWriter(saida, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    print("Gravando VIDEO... pressione 'q' na janela para PARAR.")
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            writer.write(frame)
            cv2.imshow("Captura de video (q para parar)", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        cap.release(); writer.release(); cv2.destroyAllWindows()
    return saida

# ---------- Captura de ÁUDIO (microfone) ----------
def capturar_audio(sr=16000, canais=1):
    """Grava do microfone até pressionar ENTER; salva um WAV 16k mono e retorna o caminho."""
    import sounddevice as sd
    saida = os.path.join(PASTA_CAPTURAS,
                         "audio_%s.wav" % datetime.datetime.now().strftime("%Y%m%d_%H%M%S"))
    frames = []
    def _cb(ind, out, t, status):
        frames.append(ind.copy())
    print("Gravando AUDIO... pressione ENTER para PARAR.")
    with sd.InputStream(samplerate=sr, channels=canais, dtype="int16", callback=_cb):
        input()  # bloqueia ate o usuario pressionar Enter
    dados = np.concatenate(frames) if frames else np.zeros((0, canais), np.int16)
    with wave.open(saida, "wb") as wf:
        wf.setnchannels(canais); wf.setsampwidth(2); wf.setframerate(sr)
        wf.writeframes(dados.tobytes())
    return saida

# ---------- STT local (offline) com Vosk pt-BR ----------
VOSK_DIR = os.path.join("modelos", "vosk-pt")
def _baixar_vosk(url="https://alphacephei.com/vosk/models/vosk-model-small-pt-0.3.zip"):
    """Baixa e descompacta o modelo pt-BR do Vosk em modelos/vosk-pt.
    Usa requests/certifi para contornar o erro de SSL do macOS
    (CERTIFICATE_VERIFY_FAILED); se nao der, indica download manual."""
    import zipfile
    os.makedirs("modelos", exist_ok=True)
    zipbuf = os.path.join("modelos", "vosk-pt.zip")
    baixou = False
    try:
        print("Baixando modelo Vosk pt-BR (uma vez)...")
        try:
            import requests, certifi
            with requests.get(url, stream=True, verify=certifi.where(), timeout=180) as r:
                r.raise_for_status()
                with open(zipbuf, "wb") as f:
                    for pedaco in r.iter_content(chunk_size=1 << 20):
                        if pedaco:
                            f.write(pedaco)
            baixou = True
        except ImportError:
            import urllib.request, ssl, certifi
            ctx = ssl.create_default_context(cafile=certifi.where())
            with urllib.request.urlopen(url, context=ctx) as resp, open(zipbuf, "wb") as f:
                f.write(resp.read())
            baixou = True
    except Exception as e:
        print("Falha ao baixar modelo Vosk:", e)
    if baixou:
        try:
            with zipfile.ZipFile(zipbuf) as z:
                raiz = z.namelist()[0].split("/")[0]
                z.extractall("modelos")
            destino = os.path.join("modelos", raiz)
            if not os.path.isdir(VOSK_DIR):
                os.replace(destino, VOSK_DIR)
            print("Modelo Vosk pronto em", VOSK_DIR)
        except Exception as e:
            print("Falha ao extrair o modelo Vosk:", e)
    if not os.path.isdir(VOSK_DIR):
        print("Alternativa: baixe manualmente", url)
        print("  e extraia o conteudo para a pasta:", VOSK_DIR)

def transcrever_audio_offline(caminho_audio):
    """Transcreve um WAV 16k mono usando o Vosk local (sem nuvem). Retorna '' se indisponivel."""
    try:
        import json as _json
        from vosk import Model, KaldiRecognizer
    except Exception as e:
        registrar_api("Vosk (offline)", {"status": "indisponivel", "erro": str(e)})
        print("Vosk nao instalado -> sem transcricao offline real (pip install vosk).")
        return ""
    if not os.path.isdir(VOSK_DIR):
        _baixar_vosk()
    if not os.path.isdir(VOSK_DIR):
        return ""
    model = Model(VOSK_DIR)
    rec = KaldiRecognizer(model, 16000)
    pedacos = []
    with wave.open(caminho_audio, "rb") as wf:
        if wf.getnchannels() != 1 or wf.getsampwidth() != 2:
            print("Vosk espera WAV 16kHz mono PCM; arquivo pode nao transcrever bem.")
        while True:
            dados = wf.readframes(4000)
            if not dados:
                break
            if rec.AcceptWaveform(dados):
                pedacos.append(_json.loads(rec.Result()).get("text", ""))
    pedacos.append(_json.loads(rec.FinalResult()).get("text", ""))
    texto_final = " ".join(p for p in pedacos if p).strip()
    registrar_api("Vosk (offline)", {"status": "ok", "arquivo": caminho_audio,
                                     "segmentos": len(pedacos), "texto": texto_final})
    return texto_final

# ---------- Selecao da fonte e resolucao dos caminhos ----------
def _escolher_fonte(nome):
    try:
        r = input("[%s] 1=captura  2=inserir caminho  3=simulado (Enter=3): " % nome).strip()
    except Exception:
        r = ""  # headless -> simulado
    return r or "3"

def resolver_fonte_video():
    global CAMINHO_VIDEO
    e = _escolher_fonte("VIDEO")
    CAMINHO_VIDEO = ""
    if e == "1":
        try:
            CAMINHO_VIDEO = capturar_video()
            print("Video capturado em:", CAMINHO_VIDEO)
        except Exception as err:
            print("Falha na captura de video -> simulado.", err); CAMINHO_VIDEO = ""
    elif e == "2":
        p = input("  caminho do arquivo de video (.mp4): ").strip()
        CAMINHO_VIDEO = p if p and os.path.exists(p) else ""
        if not CAMINHO_VIDEO:
            print("Arquivo nao encontrado -> simulado.")

def resolver_fonte_audio():
    global CAMINHO_AUDIO, TEXTO_AUDIO_CAPTURADO
    TEXTO_AUDIO_CAPTURADO = ""
    e = _escolher_fonte("AUDIO")
    CAMINHO_AUDIO = ""
    if e == "1":
        try:
            CAMINHO_AUDIO = capturar_audio()
            print("Audio capturado em:", CAMINHO_AUDIO)
        except Exception as err:
            print("Falha na captura de audio -> simulado.", err); CAMINHO_AUDIO = ""
    elif e == "2":
        p = input("  caminho do arquivo de audio (.wav): ").strip()
        CAMINHO_AUDIO = p if p and os.path.exists(p) else ""
        if not CAMINHO_AUDIO:
            print("Arquivo nao encontrado -> simulado.")
    # Se houver arquivo real, transcreve de verdade ja aqui:
    if CAMINHO_AUDIO:
        t = transcrever_audio(CAMINHO_AUDIO) if _azure_on else transcrever_audio_offline(CAMINHO_AUDIO)
        if t:
            TEXTO_AUDIO_CAPTURADO = t
            print("Transcricao:", TEXTO_AUDIO_CAPTURADO)

resolver_fonte_video()
resolver_fonte_audio()
print("Fontes -> video:", CAMINHO_VIDEO or "(simulado)", "| audio:", CAMINHO_AUDIO or "(simulado)")

## 4. Execução do Pipeline Multimodal (vídeo + áudio + anomalias)
Gera `resultado_video`, `resultado_audio` e `resultado_sinais` que alimentam o relatório/decisão final.

In [ ]:
# =====================================================================
# EXECUÇÃO — ANÁLISE DE VÍDEO
# Processa o vídeo real (CAMINHO_VIDEO) frame a frame ou, offline, frames
# sintéticos. Além dos alertas por frame, coleta o histórico de keypoints de
# postura (keypoints_history) que será usado pela análise de movimentação.
# =====================================================================
def gerar_frames_sinteticos(n=40):
    """Frames simulados para rodar YOLO/MediaPipe sem arquivo real."""
    frames = []
    for i in range(n):
        base = np.full((480, 640, 3), 60 + i, dtype=np.uint8)
        frames.append(base)
    return frames

def executar_video():
    resultados = None
    if os.path.exists(CAMINHO_VIDEO):  # vídeo clínico real disponível?
        try:
            resultados = processar_video_clinico(CAMINHO_VIDEO)
        except Exception as e:
            print("Falha no video real, usando frames sinteticos:", e)
            resultados = None
    if not resultados:
        # Fallback offline: aplica os MESMOS detectores sobre frames simulados
        resultados = []
        for idx, frame in enumerate(gerar_frames_sinteticos()):
            objetos = detectar_objetos_yolo(frame)
            postura = detectar_postura(frame)
            alertas, score = [], 0
            kpts = None
            if postura.get('status') == 'sucesso':
                kpts = postura.get('keypoints', {})
                lje = kpts.get('ljoelho', (0, 0, 0)); rje = kpts.get('rjoelho', (0, 0, 0))
                if lje[2] > 0.8 or rje[2] > 0.8:
                    score += 1; alertas.append('Joelho esticado')
                if lje[2] < 0.2 or rje[2] < 0.2:
                    score += 1; alertas.append('Joelho dobrado excessivamente')
            if objetos:
                labels = [o['label'] for o in objetos if o['confidence'] > 0.5]
                if labels:
                    score += len(labels)
                    alertas.append('Objetos detectados: ' + ', '.join(labels[:3]))
            resultados.append({'frame': idx, 'objetos': len(objetos),
                               'postura_ok': postura.get('status') == 'sucesso',
                               'score_anomalia': score,
                               'alertas': '; '.join(alertas) if alertas else 'Nenhum',
                               'kpts': kpts})
    # Agrega os resultados por frame nos campos usados pela integração
    alertas_listas = [r['alertas'] for r in resultados if r['alertas'] != 'Nenhum']
    keypoints_history = [r['kpts'] for r in resultados if r.get('kpts')]
    return {'frames': len(resultados),
            'objetos_totais': sum(r['objetos'] for r in resultados),
            'anomalias_posturais': sum(r['score_anomalia'] for r in resultados),
            'alertas_listas': alertas_listas,
            'total_alertas': len(alertas_listas),
            'keypoints_history': keypoints_history}

resultado_video = executar_video()
print("Video: {} frames | {} alertas | {} frames c/ postura (keypoints)".format(
    resultado_video['frames'], resultado_video['total_alertas'],
    len(resultado_video['keypoints_history'])))

In [ ]:
# =====================================================================
# EXECUÇÃO — ANÁLISE DE ÁUDIO
# Usa o texto já transcrito na seção de captura (arquivo real via Azure ou
# Vosk). Sem arquivo real (ou sem STT), cai na transcrição simulada.
# Depois analisa sentimento/entidades e detecta alterações vocais.
# =====================================================================
def executar_audio():
    # 1) texto já transcrito na captura/seleção de fonte
    texto = globals().get("TEXTO_AUDIO_CAPTURADO", "")
    # 2) plano B: se tem arquivo mas ainda não transcreveu (ex.: mudou caminho)
    if not texto and CAMINHO_AUDIO and os.path.exists(CAMINHO_AUDIO):
        texto = transcrever_audio(CAMINHO_AUDIO) if _azure_on else transcrever_audio_offline(CAMINHO_AUDIO)
    # 3) fallback: transcrição simulada (modo offline sem gravação)
    if not texto:
        texto = ("Paciente: estou muito cansado e com falta de ar. "
                 "Sinto dor no peito desde ontem.")
    analise = analisar_texto_azure(texto)
    alteracoes = detectar_alteracoes_vocais(texto)
    return {'transcricao': texto,
            'sentimento': analise.get('sentimento', {}),
            'entidades': analise.get('entidades', []),
            'alteracoes': alteracoes,
            'total_alteracoes': sum(1 for v in alteracoes.values() if v.get('presente'))}

resultado_audio = executar_audio()
print("Audio: sentimento={} | {} alteracoes vocais".format(
    resultado_audio['sentimento'].get('rotulo', 'neutro'),
    resultado_audio['total_alteracoes']))

In [ ]:
# =====================================================================
# DADOS DO PACIENTE (entrada real) + sintetizador de movimento
# - sinais vitais: dados/sinais.csv (colunas: timestamp,fc,spo2)
# - prescrições:   dados/prescricoes.json (lista de {medicamento: dosagem})
# Se o arquivo não existir, usa uma série de EXEMPLO (claramente rotulada).
# ger_keypoints_sinteticos: trajetória variada p/ quando o vídeo não fornece
# keypoints (sem câmera), para a análise de movimentação não ficar estática.
# =====================================================================
import json as _json

ARQ_SINAIS = os.path.join("dados", "sinais.csv")
ARQ_PRESCRICOES = os.path.join("dados", "prescricoes.json")

def carregar_sinais_vitais(caminho=ARQ_SINAIS):
    """Retorna (DataFrame, fonte). Se não houver CSV, usa série de exemplo."""
    if caminho and os.path.exists(caminho):
        try:
            df = pd.read_csv(caminho)
            if {"fc", "spo2"}.issubset(df.columns) and len(df) >= 1:
                return df, "dataset(%s)" % caminho
        except Exception as e:
            print("Falha ao ler sinais.csv:", e)
    df = pd.DataFrame({"fc": [72, 75, 74, 78, 80, 82, 130],
                       "spo2": [98, 98, 97, 96, 96, 95, 92]})
    return df, "EXEMPLO"

def carregar_prescricoes(caminho=ARQ_PRESCRICOES):
    """Retorna (lista de prescrições, fonte). Se não houver JSON, usa exemplo."""
    if caminho and os.path.exists(caminho):
        try:
            with open(caminho, encoding="utf-8") as f:
                dados = _json.load(f)
            if isinstance(dados, list) and dados:
                return dados, "dataset(%s)" % caminho
        except Exception as e:
            print("Falha ao ler prescricoes.json:", e)
    return ([{"Dipirona": "500mg"}, {"Dipirona": "1g", "Oxigenio": "2L/min"}], "EXEMPLO")

def gerar_keypoints_sinteticos(n=12, semente=None):
    """Caminhada do quadril com um salto brusco no meio (movimento variado)."""
    rng = np.random.default_rng(semente)
    hist, x, y = [], 0.5, 0.5
    for i in range(n):
        x = float(np.clip(x + rng.normal(0, 0.02), 0, 1))
        y = float(np.clip(y + rng.normal(0, 0.02), 0, 1))
        if i == n // 2:                      # movimento repentino
            x = float(np.clip(x + 0.6, 0, 1)); y = float(np.clip(y + 0.5, 0, 1))
        hist.append({"lquadril": (x, y), "rquadril": (x + 0.1, y)})
    return hist

In [ ]:
# =====================================================================
# EXECUÇÃO — DETECÇÃO DE ANOMALIAS (agora orientada a dados reais)
# Cada detector recebe dados REAIS quando disponíveis e registra a FONTE:
#  - sinais vitais  -> dados/sinais.csv (senão, EXEMPLO)
#  - prescrições     -> dados/prescricoes.json (senão, EXEMPLO)
#  - movimentação    -> keypoints reais do VÍDEO (senão, SINTETICO variado)
# =====================================================================
def executar_sinais():
    fontes = {}
    # 1) sinais vitais
    df_sinais, fontes["sinais_vitais"] = carregar_sinais_vitais()
    resultado_sinais = detectar_anomalias_sinais_vitais(df_sinais)
    # 2) prescrições
    presc, fontes["prescricoes"] = carregar_prescricoes()
    prescricoes = detectar_anomalias_prescricoes(presc)
    # 3) movimentação: prioriza os keypoints vindos do vídeo; senão sintetiza
    kp = resultado_video.get("keypoints_history") or []
    if len(kp) >= 2:
        fontes["movimentacao"] = "VIDEO (keypoints reais)"
    else:
        _s = os.getenv("SEMENTE", "").strip()
        kp = gerar_keypoints_sinteticos(semente=int(_s) if _s.isdigit() else None)
        fontes["movimentacao"] = "SINTETICO (sem keypoints do video)"
    movimentacao = detectar_anomalias_movimentacao(kp)
    # consolida alertas das três fontes
    alertas = list(resultado_sinais.get("alertas", []))
    alertas += prescricoes.get("alertas", []) + movimentacao.get("alertas", [])
    resultado_sinais.update({"prescricoes": prescricoes,
                             "movimentacao": movimentacao,
                             "alertas": alertas,
                             "fontes": fontes,
                             "n_keypoints_movimentacao": len(kp)})
    registrar_api("anomalias/fontes", fontes)
    return resultado_sinais

resultado_sinais = executar_sinais()
print("Sinais/anomalias ->", resultado_sinais["alertas"])
print("  fontes ->", resultado_sinais["fontes"])

# Relatório Final para Apresentacao de 15 Minutos

In [ ]:
# =====================================================================
# RELATÓRIO FINAL (para a apresentação de 15 minutos)
# Resume os resultados das três análises e produz a decisão clínica final
# chamando o workflow_integrado com tudo que foi processado.
# =====================================================================
print('=' * 70)
print('TECH CHALLENGE IADT FASE 4 - SISTEMA MULTIMODAL COMPLETO')
print('Modo de execucao:', MODO.upper())
print('=' * 70)

# --- Resumo da ANÁLISE DE VÍDEO ---
print('\n## 1. DEMONSTRAÇÃO DE ANÁLISE DE VÍDEO')
print('   • OpenPose-Light/MediaPipe: analise postural em {} frames'.format(resultado_video['frames']))
print('   • YOLOv8: {} objetos/areas criticos detectados'.format(resultado_video['objetos_totais']))
print('   • Anomalias posturais (score): {}'.format(resultado_video['anomalias_posturais']))
print('   • Alertas de video: {}'.format(resultado_video['total_alertas']))

# --- Resumo da ANÁLISE DE ÁUDIO ---
print('\n## 2. DEMONSTRAÇÃO DE ANÁLISE DE ÁUDIO')
print('   • Transcricao: "{}"'.format(resultado_audio['transcricao'][:80]))
print('   • Sentimento: {}'.format(resultado_audio['sentimento'].get('rotulo', 'neutro')))
print('   • Entidades medicas: {}'.format(len(resultado_audio['entidades'])))
print('   • Alteracoes vocais: {}'.format(resultado_audio['total_alteracoes']))

# --- Resumo da DETECÇÃO DE ANOMALIAS ---
print('\n## 3. DETECÇÃO DE ANOMALIAS')
print('   • Sinais/prescricoes/movimentacao ->', '; '.join(resultado_sinais.get('alertas', ['nenhuma'])))
print('   • Fontes ->', resultado_sinais.get('fontes', {}))

# --- INTEGRAÇÃO: gera o alerta final ---
print('\n## 4. INTEGRAÇÃO E DECISÃO FINAL')
resultado = workflow_integrado(resultado_video, resultado_audio, resultado_sinais)
print('   • Decisao:', resultado['decisao'])
print('   • Nivel:', resultado['nivel'])
print('   • Acao:', resultado['acao_recomendada'])
print('   • Alertas gerados:', resultado['total_alertas'])
print('   • Pontuacao de risco:', resultado['punteis_risco'])
print('   • Detalhe:', resultado['alertas'])

print('\n' + '=' * 70)
print('FIM DA APRESENTAÇÃO DO TECH CHALLENGE IADT FASE 4')
print('=' * 70)

In [ ]:
# =====================================================================
# RESUMO CONSOLIDADO — todos os retornos das APIs/passo (para auditoria)
# =====================================================================
registrar_api("resultado_video", resultado_video)
registrar_api("resultado_audio", resultado_audio)
registrar_api("resultado_sinais", resultado_sinais)
registrar_api("decisao_final", resultado)

print("\n" + "=" * 70)
print("LOG DE RETORNOS (APIs e etapas) — total: %d" % len(REGISTRO_API))
print("=" * 70)
for i, r in enumerate(REGISTRO_API, 1):
    print("%02d. %-30s -> %s" % (i, r["nome"], _resumir(r["dado"], 500)))
print("-" * 70)
print("Log completo tambem em:", LOG_FILE)